In [4]:
import numpy as np
import matplotlib.pyplot as plt
import time

N_pmts = 40
R = 2.0
r = 0.1
indices = np.arange(0, N_pmts, dtype=float) + 0.5
phi = np.arccos(1 - 2 * indices / N_pmts)
theta = np.pi * (1 + 5**0.5) * indices
x = np.cos(theta) * np.sin(phi) * R
y = np.sin(theta) * np.sin(phi) * R
z = np.cos(phi) * R

PMT_POSITIONS = np.column_stack([x, y, z])

def simulate_event_slow(N_photons):
    photons = []
    for _ in range(N_photons):
        nx, ny, nz = np.random.normal(size=3)
        norm = np.sqrt(nx**2 + ny**2 + nz**2)
        photons.append((nx/norm, ny/norm, nz/norm))
        
    hits = [0] * N_pmts
    

    for nx, ny, nz in photons:
        hx, hy, hz = nx * R, ny * R, nz * R
        
        for i in range(N_pmts):
            dist = np.sqrt((hx - PMT_POSITIONS[i, 0])**2 + 
                           (hy - PMT_POSITIONS[i, 1])**2 + 
                           (hz - PMT_POSITIONS[i, 2])**2)

    total_pe = 0
    for h in hits:
        total_pe += np.random.poisson(0.5 * h) # a=0.5
        
    return total_pe

def simulate_event_fast(N_photons):
    photons = np.random.normal(size=(N_photons, 3))
    norms = np.linalg.norm(photons, axis=1, keepdims=True)
    photons = photons / norms 
    hit_points = photons * R  

    diffs = hit_points[:, np.newaxis, :] - PMT_POSITIONS[np.newaxis, :, :]
    dists = np.linalg.norm(diffs, axis=2)
    hits_array = np.sum(dists <= r, axis=0)  
    total_pe = np.sum(np.random.poisson(0.5 * hits_array))
    
    return total_pe

N_photons_to_test = 10000 
N_events = 50               

print(f"Симуляция {N_events} событий по {N_photons_to_test} фотонов")


start_time = time.time()
for _ in range(N_events):
    simulate_event_slow(N_photons_to_test)
time_slow = time.time() - start_time

start_time = time.time()
for _ in range(N_events):
    simulate_event_fast(N_photons_to_test)
time_fast = time.time() - start_time


print(f"\nВремя медленного способа (list + for): {time_slow:.4f} сек.")
print(f"Время быстрого способа (numpy.array):  {time_fast:.4f} сек.")
print(f"Ускорение: {time_slow / time_fast:.1f} раз")

Симуляция 50 событий по 10000 фотонов

Время медленного способа (list + for): 56.8126 сек.
Время быстрого способа (numpy.array):  1.0588 сек.
Ускорение: 53.7 раз
